In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName('spark_employee')\
        .config('spark.sql.shuffle.partitions',2)\
        .getOrCreate()
spark

In [5]:
df_employee_data = spark.read.csv('/content/sample_data/employees_data.csv',header = True,inferSchema=True)
df_employee_data.show()
df_employee_data.printSchema()
df_employee_data.count()

+-----------+-------+---+---------+------+----------+
|customer_id|   name|age|     city|salary|department|
+-----------+-------+---+---------+------+----------+
|          1|  Sunil| 28|Bangalore| 55000|        IT|
|          2|   Ravi| 35|Hyderabad| 72000|   Finance|
|          3|  Anita| 24|Bangalore| 40000|        HR|
|          4|  Kiran| 42|  Chennai| 85000|        IT|
|          5|  Meena| 30|Hyderabad|  NULL|   Finance|
|          6|  Arjun| 27|Bangalore| 48000|        IT|
|          7|  Pooja| 33|   Mumbai| 62000|        HR|
|          8| Vikram| 45|  Chennai| 90000|Management|
|          9|   Neha| 29|Bangalore|  NULL|        IT|
|         10|   Amit| 38|   Mumbai| 70000|   Finance|
|         11|  Sneha| 26|Hyderabad| 45000|        HR|
|         12|  Rahul| 31|Bangalore| 52000|        IT|
|         13|  Kavya| 23|  Chennai| 38000|        HR|
|         14|  Manoj| 41|   Mumbai| 88000|Management|
|         15|  Divya| 36|Bangalore| 67000|   Finance|
|         16|   Ajay| 28|Hyd

20

In [6]:
from pyspark.sql.functions import col, sum

null_count = df_employee_data.select([sum(col(c).isNull().cast('int')).alias(c) for c in df_employee_data.columns])

null_cols_with_count = {
         col_name: count for col_name, count in null_count.first().asDict().items() if count > 0
}

for col_name, count in null_cols_with_count.items():
  print(f'column {col_name} has {count} null values.')

column salary has 3 null values.


In [7]:
df_employee_data.dropna(subset='customer_id').count()

20

In [8]:
df_employee_data.filter(df_employee_data['salary'] > 50000).show()
df_employee_data.filter(df_employee_data['city'] == 'Bangalore').show()
df_employee_data.filter((df_employee_data['age'] >= 25)
                         & (df_employee_data['age'] <= 35)).show()

+-----------+-------+---+---------+------+----------+
|customer_id|   name|age|     city|salary|department|
+-----------+-------+---+---------+------+----------+
|          1|  Sunil| 28|Bangalore| 55000|        IT|
|          2|   Ravi| 35|Hyderabad| 72000|   Finance|
|          4|  Kiran| 42|  Chennai| 85000|        IT|
|          7|  Pooja| 33|   Mumbai| 62000|        HR|
|          8| Vikram| 45|  Chennai| 90000|Management|
|         10|   Amit| 38|   Mumbai| 70000|   Finance|
|         12|  Rahul| 31|Bangalore| 52000|        IT|
|         14|  Manoj| 41|   Mumbai| 88000|Management|
|         15|  Divya| 36|Bangalore| 67000|   Finance|
|         17|  Nisha| 34|  Chennai| 61000|        HR|
|         20|Lakshmi| 32|Hyderabad| 58000|        HR|
+-----------+-------+---+---------+------+----------+

+-----------+-----+---+---------+------+----------+
|customer_id| name|age|     city|salary|department|
+-----------+-----+---+---------+------+----------+
|          1|Sunil| 28|Bangalore|

In [9]:
from pyspark.sql.functions import col, when

df_employee_data.withColumn(
    'salary_category',
    when(col('salary') > 70000, 'High')
    .when((col('salary') > 40000) & (col('salary') < 70000), 'Medium')
    .when(col('salary') < 40000, 'Low')
    .otherwise(None)
).show(10)

df_employee_data.withColumn(
    'age_group',
    when(col('age') >= 40, 'Senior')
    .when((col('age') < 40) & (col('age') > 25), 'Audult')
    .when(col('age') < 25, 'Young')
).show(5)

+-----------+------+---+---------+------+----------+---------------+
|customer_id|  name|age|     city|salary|department|salary_category|
+-----------+------+---+---------+------+----------+---------------+
|          1| Sunil| 28|Bangalore| 55000|        IT|         Medium|
|          2|  Ravi| 35|Hyderabad| 72000|   Finance|           High|
|          3| Anita| 24|Bangalore| 40000|        HR|           NULL|
|          4| Kiran| 42|  Chennai| 85000|        IT|           High|
|          5| Meena| 30|Hyderabad|  NULL|   Finance|           NULL|
|          6| Arjun| 27|Bangalore| 48000|        IT|         Medium|
|          7| Pooja| 33|   Mumbai| 62000|        HR|         Medium|
|          8|Vikram| 45|  Chennai| 90000|Management|           High|
|          9|  Neha| 29|Bangalore|  NULL|        IT|           NULL|
|         10|  Amit| 38|   Mumbai| 70000|   Finance|           NULL|
+-----------+------+---+---------+------+----------+---------------+
only showing top 10 rows
+--------

In [10]:
from pyspark.sql.functions import avg, round, count, min, max
df_employee_data.groupBy('department').agg(round(avg('salary'),2).alias('avg_salary')).show()
df_employee_data.groupBy('city').agg(count('customer_id').alias('employee_count')).show()
df_employee_data.groupBy('department').agg(min('salary').alias('min_salary')).show()
df_employee_data.groupBy('department').agg(max('salary').alias('max_salary')).show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|        IT|  55166.67|
|   Finance|  69666.67|
|        HR|  50666.67|
|Management|   89000.0|
+----------+----------+

+---------+--------------+
|     city|employee_count|
+---------+--------------+
|Bangalore|             7|
|Hyderabad|             5|
|  Chennai|             4|
|   Mumbai|             4|
+---------+--------------+

+----------+----------+
|department|min_salary|
+----------+----------+
|        IT|     42000|
|   Finance|     67000|
|        HR|     38000|
|Management|     88000|
+----------+----------+

+----------+----------+
|department|max_salary|
+----------+----------+
|        IT|     85000|
|   Finance|     72000|
|        HR|     62000|
|Management|     90000|
+----------+----------+



In [11]:
df_employee_data.sort('salary', descending = True).show()
df_employee_data.sort('salary',descending = True).show(5)
df_employee_data.sort('salary',descending = False).show(5)

+-----------+-------+---+---------+------+----------+
|customer_id|   name|age|     city|salary|department|
+-----------+-------+---+---------+------+----------+
|          5|  Meena| 30|Hyderabad|  NULL|   Finance|
|          9|   Neha| 29|Bangalore|  NULL|        IT|
|         18|  Varun| 39|   Mumbai|  NULL|   Finance|
|         13|  Kavya| 23|  Chennai| 38000|        HR|
|          3|  Anita| 24|Bangalore| 40000|        HR|
|         19|   Teja| 25|Bangalore| 42000|        IT|
|         11|  Sneha| 26|Hyderabad| 45000|        HR|
|          6|  Arjun| 27|Bangalore| 48000|        IT|
|         16|   Ajay| 28|Hyderabad| 49000|        IT|
|         12|  Rahul| 31|Bangalore| 52000|        IT|
|          1|  Sunil| 28|Bangalore| 55000|        IT|
|         20|Lakshmi| 32|Hyderabad| 58000|        HR|
|         17|  Nisha| 34|  Chennai| 61000|        HR|
|          7|  Pooja| 33|   Mumbai| 62000|        HR|
|         15|  Divya| 36|Bangalore| 67000|   Finance|
|         10|   Amit| 38|   

In [12]:
from pyspark.sql.functions import mean

mean_salary = df_employee_data.select(mean('salary')).collect()[0][0]

salary_nulls_replaced = df_employee_data.fillna(subset= 'salary', value = mean_salary)
salary_nulls_replaced.count()

20

In [13]:
from pyspark.sql.functions import col, sum

null_vals = salary_nulls_replaced.select([sum(col(c).isNull().cast('int').alias(c)) for c in salary_nulls_replaced.columns])

null_vals_only = {
    col_name: count for col_name, count in null_vals.first().asDict().items() if count > 0
}

for col_name, count in null_vals_only.items():
  print(f'The column {col_name} has {count} null values')


In [14]:
from pyspark.sql.functions import col, count, mean

# mean_salary is already computed in cell 03ffe194
salary_nulls_replaced.filter(col('salary') > mean_salary).show()

employee_count = salary_nulls_replaced.distinct().count()

salary_nulls_replaced.groupBy('city').agg(count('customer_id').alias('employee_count_in_city')).filter(col('employee_count_in_city') > 3).show()

+-----------+------+---+---------+------+----------+
|customer_id|  name|age|     city|salary|department|
+-----------+------+---+---------+------+----------+
|          2|  Ravi| 35|Hyderabad| 72000|   Finance|
|          4| Kiran| 42|  Chennai| 85000|        IT|
|          7| Pooja| 33|   Mumbai| 62000|        HR|
|          8|Vikram| 45|  Chennai| 90000|Management|
|         10|  Amit| 38|   Mumbai| 70000|   Finance|
|         14| Manoj| 41|   Mumbai| 88000|Management|
|         15| Divya| 36|Bangalore| 67000|   Finance|
|         17| Nisha| 34|  Chennai| 61000|        HR|
+-----------+------+---+---------+------+----------+

+---------+----------------------+
|     city|employee_count_in_city|
+---------+----------------------+
|Bangalore|                     7|
|Hyderabad|                     5|
|  Chennai|                     4|
|   Mumbai|                     4|
+---------+----------------------+



In [15]:
from pyspark.sql.functions import col, count

dup_records = salary_nulls_replaced.groupBy('name', 'city').agg(count('customer_id').alias('customer_count')).filter(col('customer_count') > 1)
dup_records.show()

+----+----+--------------+
|name|city|customer_count|
+----+----+--------------+
+----+----+--------------+



In [16]:
salary_nulls_replaced.write.csv(f'/content/sample_data/employee_transformed')

In [17]:
unique_employees = salary_nulls_replaced.distinct()
print(f"Number of unique employees: {unique_employees.count()}")
unique_employees.show()

Number of unique employees: 20
+-----------+-------+---+---------+------+----------+
|customer_id|   name|age|     city|salary|department|
+-----------+-------+---+---------+------+----------+
|          1|  Sunil| 28|Bangalore| 55000|        IT|
|          2|   Ravi| 35|Hyderabad| 72000|   Finance|
|          3|  Anita| 24|Bangalore| 40000|        HR|
|          6|  Arjun| 27|Bangalore| 48000|        IT|
|          7|  Pooja| 33|   Mumbai| 62000|        HR|
|          9|   Neha| 29|Bangalore| 60117|        IT|
|         11|  Sneha| 26|Hyderabad| 45000|        HR|
|         12|  Rahul| 31|Bangalore| 52000|        IT|
|         14|  Manoj| 41|   Mumbai| 88000|Management|
|         16|   Ajay| 28|Hyderabad| 49000|        IT|
|         17|  Nisha| 34|  Chennai| 61000|        HR|
|         18|  Varun| 39|   Mumbai| 60117|   Finance|
|          4|  Kiran| 42|  Chennai| 85000|        IT|
|          5|  Meena| 30|Hyderabad| 60117|   Finance|
|          8| Vikram| 45|  Chennai| 90000|Managemen

In [18]:
unique_employee_ids = salary_nulls_replaced.select('customer_id').distinct()
print(f"Number of unique employee IDs: {unique_employee_ids.count()}")
unique_employee_ids.show()

Number of unique employee IDs: 20
+-----------+
|customer_id|
+-----------+
|          2|
|          4|
|          5|
|         10|
|         12|
|         13|
|         14|
|         18|
|          1|
|          3|
|          6|
|          7|
|          8|
|          9|
|         11|
|         15|
|         16|
|         17|
|         19|
|         20|
+-----------+



In [ ]:
avg_df = df_employee_data.groupBy().agg(mean('salary').alias('avg_salary'))

df_employee_data.join(avg_df).filter(col('salary') > col('avg_salary')).show()

In [28]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lead, lag, rank, dense_rank
from pyspark.sql.functions import sum,col

window_spec = Window.orderBy('salary')
salary_nulls_replaced = salary_nulls_replaced.withColumn('Running_total', sum('salary').over(window_spec))
# salary_nulls_replaced.show()

window_spec02 = Window.partitionBy('department').orderBy(col('salary').desc())
salary_nulls_replaced = salary_nulls_replaced.withColumn('sal_rank_on_dept', rank().over(window_spec02))
# salary_nulls_replaced.show()

salary_nulls_replaced = salary_nulls_replaced.withColumn('sal_dens_rank', dense_rank().over(window_spec02))
# salary_nulls_replaced.show()

salary_nulls_replaced = salary_nulls_replaced.withColumn('sal_row_number', row_number().over(window_spec02))
# salary_nulls_replaced.show()

salary_nulls_replaced = salary_nulls_replaced.withColumn('sal_lag', lag('salary').over(window_spec02))
# salary_nulls_replaced.show()

salary_nulls_replaced = salary_nulls_replaced.withColumn('sal_lead', lead('salary').over(window_spec02))
salary_nulls_replaced.show()

+-----------+-------+---+---------+------+----------+-------------+----------------+-------------+--------------+-------+--------+
|customer_id|   name|age|     city|salary|department|Running_total|sal_rank_on_dept|sal_dens_rank|sal_row_number|sal_lag|sal_lead|
+-----------+-------+---+---------+------+----------+-------------+----------------+-------------+--------------+-------+--------+
|          2|   Ravi| 35|Hyderabad| 72000|   Finance|       939351|               1|            1|             1|   NULL|   70000|
|         10|   Amit| 38|   Mumbai| 70000|   Finance|       867351|               2|            2|             2|  72000|   67000|
|         15|  Divya| 36|Bangalore| 67000|   Finance|       797351|               3|            3|             3|  70000|   60117|
|          5|  Meena| 30|Hyderabad| 60117|   Finance|       607351|               4|            4|             4|  67000|   60117|
|         18|  Varun| 39|   Mumbai| 60117|   Finance|       607351|               4

In [30]:
df_departments = spark.read.csv("/content/sample_data/department_data.csv", header=True, inferSchema=True)
df_projects = spark.read.csv("/content/sample_data/projects.csv", header=True, inferSchema=True)
df_emp_projects = spark.read.csv("/content/sample_data/employee_projects.csv", header=True, inferSchema=True)

df_departments.show()
df_projects.show()
df_emp_projects.show()

+----------+------------+------+
|department|manager_name|budget|
+----------+------------+------+
|        IT|      Rajesh|500000|
|        HR|       Anita|200000|
|   Finance|      Suresh|300000|
|Management|     Kavitha|800000|
+----------+------------+------+

+----------+-----------------+----------+--------------+
|project_id|     project_name|department|project_budget|
+----------+-----------------+----------+--------------+
|        P1|   Data Migration|        IT|        200000|
|        P2|    HR Automation|        HR|        100000|
|        P3|Finance Dashboard|   Finance|        150000|
|        P4|      AI Platform|        IT|        300000|
|        P5| Company Strategy|Management|        400000|
+----------+-----------------+----------+--------------+

+-----------+----------+
|customer_id|project_id|
+-----------+----------+
|          1|        P1|
|          1|        P4|
|          2|        P3|
|          3|        P1|
|          4|        P4|
|          5|        

In [36]:
from pyspark.sql.functions import col

df_emp = salary_nulls_replaced.alias('emp')
df_dept = df_departments.alias('dept')
df_proj = df_projects.alias('proj')
df_emp_prj = df_emp_projects.alias('emp_prj')

result = df_emp \
         .join(df_emp_prj, col('emp.customer_id') == col('emp_prj.customer_id'),'left')\
         .join( df_proj, col('emp_prj.project_id') == col('proj.project_id'), 'inner') \
         .join(df_dept, col('proj.department') == col('dept.department'), 'inner')\
         .select(
             'emp.customer_id',
             'emp.name',
             'dept.department',
             'dept.manager_name'
         )

result.show()

+-----------+------+----------+------------+
|customer_id|  name|department|manager_name|
+-----------+------+----------+------------+
|          1| Sunil|        IT|      Rajesh|
|          1| Sunil|        IT|      Rajesh|
|          2|  Ravi|   Finance|      Suresh|
|          3| Anita|        IT|      Rajesh|
|          4| Kiran|        IT|      Rajesh|
|          5| Meena|   Finance|      Suresh|
|          6| Arjun|        IT|      Rajesh|
|          7| Pooja|        HR|       Anita|
|          8|Vikram|Management|     Kavitha|
|          9|  Neha|        IT|      Rajesh|
|         10|  Amit|   Finance|      Suresh|
|         11| Sneha|        HR|       Anita|
|         12| Rahul|        IT|      Rajesh|
|         13| Kavya|        HR|       Anita|
|         14| Manoj|Management|     Kavitha|
|         15| Divya|   Finance|      Suresh|
|         16|  Ajay|        IT|      Rajesh|
|         17| Nisha|        HR|       Anita|
|         18| Varun|   Finance|      Suresh|
|         